Q1. Write sql query to find the products whose total sales revenue has increased every year. Include product_id, product_name, category in result.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
product_data = [
 (1, 'Laptops', 'Electronics'),
 (2, 'Jeans', 'Clothing'),
 (3, 'Chairs', 'Home Appliances')
 ]

product_schema = ['product_id', 'product_name', 'category']

product_df = spark.createDataFrame(data = product_data , schema = product_schema)

In [0]:
sales_data = [
 (1, 2019, 1000.00),
 (1, 2020, 1200.00),
 (1, 2021, 1100.00),
 (2, 2019, 500.00),
 (2, 2020, 600.00),
 (2, 2021, 900.00),
 (3, 2019, 300.00),
 (3, 2020, 450.00),
 (3, 2021, 400.00)
 ]

sales_schema = ['product_id', 'year', 'total_sales_revenue']

sales_df = spark.createDataFrame(data = sales_data , schema = sales_schema)

In [0]:
w = Window.partitionBy(F.col("product_id")).orderBy(F.col("year"))
w_count = Window.partitionBy(F.col("product_id"))
df = (
    sales_df.groupby(['product_id','year'])
    .agg(
        F.sum("total_sales_revenue").alias("total_sales_revenue")
    )
    .withColumn("next_year",F.lead(F.col("year")).over(w))
    .withColumn("next_total_sales_revenue",F.lead(F.col("total_sales_revenue")).over(w))
    .withColumn("total_count",F.count(F.col("product_id")).over(w_count))
    .filter((F.col("next_year") - F.col("year") == 1) & (F.col("next_total_sales_revenue") > F.col("total_sales_revenue")))
    .withColumn("filtered_count",F.count(F.col("product_id")).over(w_count))
    .filter(F.col("total_count") - F.col("filtered_count") == 1)
    .select("product_id")
    .drop_duplicates()
)

merged_df = (
    df.join(product_df, how ='inner', on ='product_id')
)
display(merged_df)

# m2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("product_id").orderBy("year")

df = (
    sales_df
    .withColumn(
        "prev_revenue",
        F.lag("total_sales_revenue").over(w)
    )
    .withColumn(
        "is_increasing",
        F.when(
            (F.col("prev_revenue").isNull()) |
            (F.col("total_sales_revenue") > F.col("prev_revenue")),
            1
        ).otherwise(0)
    )
)

result = (
    df.groupBy("product_id")
      .agg(
          F.min("is_increasing").alias("all_increasing")
      )
      .filter(F.col("all_increasing") == 1)
      .join(product_df, "product_id")
      .select("product_id", "product_name", "category")
)

result.show()

# Q2 Flatten JSON

In [0]:
df_json = spark.read.format("json").option("multiline", True).load("/Workspace/Users/akhandpschauhan@gmail.com/Spark-learning-practice/DE_with_Dhairy/json_df.json")
display(df_json)

In [0]:
import pyspark.sql.functions as F

In [0]:
df_json.printSchema()

In [0]:
exploaded_df = df_json.select("Course_type","Head_Office_Contact","Institute_Name",F.explode(F.col("branches")).alias("branch_data"))

display(exploaded_df)

# Formula = array -> explode , sturct -> col_name

In [0]:
df = exploaded_df.select("Course_type","Head_Office_Contact","Institute_Name","branch_data.*")

display(df)